In [28]:
# import h5py
# from tensorflow import keras
# model = keras.models.load_model('Big_Model.h5')
# model.summary()

In [ ]:
# Import AI stuff and load data
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")



(train_df.shape, test_df.shape)

((2109600, 8), (451440, 8))

In [ ]:
# clean up values

def fill_with_local_mean(series):
    values = series.values.copy()

    for i in range(len(values)):
        if pd.isna(values[i]):
            neighbors = []

            # Previous 2
            if i-2 >= 0:
                neighbors.append(values[i-2])
            if i-1 >= 0:
                neighbors.append(values[i-1])

            # Next 2
            if i+1 < len(values):
                neighbors.append(values[i+1])
            if i+2 < len(values):
                neighbors.append(values[i+2])

            # Remove NaNs from neighbors
            neighbors = [x for x in neighbors if not pd.isna(x)]

            if len(neighbors) > 0:
                values[i] = np.mean(neighbors)

    return pd.Series(values, index=series.index)

train_df["oxygen_saturation"] = fill_with_local_mean(train_df["oxygen_saturation"])
train_df["respiratory_rate"] = fill_with_local_mean(train_df["respiratory_rate"])
train_df["heart_rate"] = fill_with_local_mean(train_df["heart_rate"])
train_df["systolic_bp"] = fill_with_local_mean(train_df["systolic_bp"])
train_df["diastolic_bp"] = fill_with_local_mean(train_df["diastolic_bp"])

train_df[train_df.isna().any(axis=1)]


,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index
3122,2025-01-01 23:20:10,7a0ccfea-7f8a-fc6e-35fa-3775b13950da_4,0.0,0.0,0.0,0.00,0.0,0,0.0,0.0,NaN
3123,2025-01-01 23:20:15,7a0ccfea-7f8a-fc6e-35fa-3775b13950da_4,0.0,0.0,0.0,0.00,0.0,0,0.0,0.0,NaN
3124,2025-01-01 23:20:20,7a0ccfea-7f8a-fc6e-35fa-3775b13950da_4,0.0,0.0,0.0,0.00,0.0,0,0.0,0.0,NaN
3125,2025-01-01 23:20:25,7a0ccfea-7f8a-fc6e-35fa-3775b13950da_4,0.0,0.0,0.0,0.00,0.0,0,0.0,0.0,NaN
3126,2025-01-01 23:20:30,7a0ccfea-7f8a-fc6e-35fa-3775b13950da_4,0.0,0.0,0.0,0.00,0.0,0,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2108154,2025-05-03 18:59:30,3ee6bbfa-9b63-6790-505e-6200cf762b7d_2927,0.0,0.0,0.0,1.25,0.0,3,0.0,0.0,NaN
2108155,2025-05-03 18:59:35,3ee6bbfa-9b63-6790-505e-6200cf762b7d_2927,0.0,0.0,0.0,0.00,0.0,3,0.0,0.0,NaN
2108156,2025-05-03 18:59:40,3ee6bbfa-9b63-6790-505e-6200cf762b7d_2927,0.0,0.0,0.0,5.00,0.0,3,0.0,0.0,NaN
2108157,2025-05-03 18:59:45,3ee6bbfa-9b63-6790-505e-6200cf762b7d_2927,0.0,0.0,0.0,5.00,0.0,3,0.0,0.0,NaN


In [80]:

test_df["oxygen_saturation"] = fill_with_local_mean(test_df["oxygen_saturation"])
test_df["respiratory_rate"] = fill_with_local_mean(test_df["respiratory_rate"])
test_df["heart_rate"] = fill_with_local_mean(test_df["heart_rate"])
test_df["systolic_bp"] = fill_with_local_mean(test_df["systolic_bp"])
test_df["diastolic_bp"] = fill_with_local_mean(test_df["diastolic_bp"])

test_df[test_df.isna().any(axis=1)]

,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label


In [30]:
# Get rid of any useless cols that doesnt help prediction
train_df['pulse_pressure'] = train_df['systolic_bp'] - train_df['diastolic_bp']
train_df['MAP'] = (2 * (train_df['diastolic_bp'] + train_df['systolic_bp'])) / 3 
train_df['shock_index'] = train_df['heart_rate'] / train_df['systolic_bp']
train_df['shock_index'] = train_df['shock_index'].fillna(0)


train_df.head()


,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index
0,2025-01-01 19:00:00,b317e7ee-8af7-3e9c-3e0f-646395b8c81a_0,88.0,104.0,74.0,18.0,95.0,0,30.0,118.666667,0.846154
1,2025-01-01 19:00:05,b317e7ee-8af7-3e9c-3e0f-646395b8c81a_0,88.0,100.0,72.0,18.0,95.0,0,28.0,114.666667,0.880000
2,2025-01-01 19:00:10,b317e7ee-8af7-3e9c-3e0f-646395b8c81a_0,86.0,96.0,70.0,17.0,94.0,0,26.0,110.666667,0.895833
3,2025-01-01 19:00:15,b317e7ee-8af7-3e9c-3e0f-646395b8c81a_0,89.0,98.0,70.0,18.0,94.0,0,28.0,112.000000,0.908163
4,2025-01-01 19:00:20,b317e7ee-8af7-3e9c-3e0f-646395b8c81a_0,91.0,98.0,70.0,18.5,94.0,0,28.0,112.000000,0.928571


In [81]:
test_df['pulse_pressure'] = test_df['systolic_bp'] - test_df['diastolic_bp']
test_df['MAP'] = (2 * (test_df['diastolic_bp'] + test_df['systolic_bp'])) / 3 
test_df['shock_index'] = test_df['heart_rate'] / test_df['systolic_bp']
test_df['shock_index'] = test_df['shock_index'].fillna(0)

In [31]:
# Get rid of any entries with null vals
train_df = train_df.sort_values(["encounter_id", "timestamp"])
train_df.head()

,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index
1577520,2025-04-03 02:00:00,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,75.0,110.0,58.0,15.0,97.0,0,52.0,112.000000,0.681818
1577521,2025-04-03 02:00:05,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,78.0,110.0,58.0,13.0,98.0,0,52.0,112.000000,0.709091
1577522,2025-04-03 02:00:10,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,83.0,110.0,58.0,15.0,97.0,0,52.0,112.000000,0.754545
1577523,2025-04-03 02:00:15,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,77.0,110.0,58.5,16.0,97.0,0,51.5,112.333333,0.700000
1577524,2025-04-03 02:00:20,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,79.0,108.0,58.0,17.0,97.0,0,50.0,110.666667,0.731481


In [82]:
test_df = test_df.sort_values(["encounter_id", "timestamp"])
test_df.head()

,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index
7920,2025-01-02 06:00:00,00945edc-641d-5f9d-f4b0-eaf147a61556_11,102.0,112.0,82.0,16.0,97.0,0,30.0,129.333333,0.910714
7921,2025-01-02 06:00:05,00945edc-641d-5f9d-f4b0-eaf147a61556_11,107.0,114.0,84.0,16.0,97.0,0,30.0,132.000000,0.938596
7922,2025-01-02 06:00:10,00945edc-641d-5f9d-f4b0-eaf147a61556_11,102.0,116.0,84.0,16.0,97.0,0,32.0,133.333333,0.879310
7923,2025-01-02 06:00:15,00945edc-641d-5f9d-f4b0-eaf147a61556_11,98.0,118.0,86.0,16.0,97.0,0,32.0,136.000000,0.830508
7924,2025-01-02 06:00:20,00945edc-641d-5f9d-f4b0-eaf147a61556_11,97.0,120.0,88.0,15.0,97.0,0,32.0,138.666667,0.808333


In [35]:
# add time based data
window_size = 12  # 1 minute

train_df["hr_roll_mean_60s"] = (
    train_df.groupby("encounter_id")["heart_rate"]
      .rolling(window_size, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

train_df["spo2_roll_mean_60s"] = (
    train_df.groupby("encounter_id")["oxygen_saturation"]
      .rolling(window_size, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

train_df["hr_roll_std_60s"] = (
    train_df.groupby("encounter_id")["heart_rate"]
      .rolling(window_size, min_periods=1)
      .std()
      .reset_index(level=0, drop=True)
)

train_df["hr_delta"] = (
    train_df.groupby("encounter_id")["heart_rate"]
      .diff()
)

train_df["spo2_delta"] = (
    train_df.groupby("encounter_id")["oxygen_saturation"]
      .diff()
)


def rolling_slope(series, window=12):
    slopes = [np.nan] * len(series)

    for i in range(window - 1, len(series)):
        y = series[i-window+1:i+1]
        x = np.arange(len(y))
        slope = np.polyfit(x, y, 1)[0]
        slopes[i] = slope

    return slopes

train_df["hr_slope_60s"] = (
    train_df.groupby("encounter_id")["heart_rate"]
      .transform(lambda x: rolling_slope(x.values, window=12))
)

train_df.head()

,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index,hr_roll_mean_60s,spo2_roll_mean_60s,hr_roll_std_60s,hr_delta,spo2_delta,hr_slope_60s
1577520,2025-04-03 02:00:00,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,75.0,110.0,58.0,15.0,97.0,0,52.0,112.000000,0.681818,75.000000,97.000000,NaN,NaN,NaN,NaN
1577521,2025-04-03 02:00:05,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,78.0,110.0,58.0,13.0,98.0,0,52.0,112.000000,0.709091,76.500000,97.500000,2.121320,3.0,1.0,NaN
1577522,2025-04-03 02:00:10,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,83.0,110.0,58.0,15.0,97.0,0,52.0,112.000000,0.754545,78.666667,97.333333,4.041452,5.0,-1.0,NaN
1577523,2025-04-03 02:00:15,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,77.0,110.0,58.5,16.0,97.0,0,51.5,112.333333,0.700000,78.250000,97.250000,3.403430,-6.0,0.0,NaN
1577524,2025-04-03 02:00:20,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,79.0,108.0,58.0,17.0,97.0,0,50.0,110.666667,0.731481,78.400000,97.200000,2.966479,2.0,0.0,NaN


In [83]:
test_df["hr_roll_mean_60s"] = (
    test_df.groupby("encounter_id")["heart_rate"]
      .rolling(window_size, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

test_df["spo2_roll_mean_60s"] = (
    test_df.groupby("encounter_id")["oxygen_saturation"]
      .rolling(window_size, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

test_df["hr_roll_std_60s"] = (
    test_df.groupby("encounter_id")["heart_rate"]
      .rolling(window_size, min_periods=1)
      .std()
      .reset_index(level=0, drop=True)
)

test_df["hr_delta"] = (
    test_df.groupby("encounter_id")["heart_rate"]
      .diff()
)

test_df["spo2_delta"] = (
    test_df.groupby("encounter_id")["oxygen_saturation"]
      .diff()
)


def rolling_slope(series, window=12):
    slopes = [np.nan] * len(series)

    for i in range(window - 1, len(series)):
        y = series[i-window+1:i+1]
        x = np.arange(len(y))
        slope = np.polyfit(x, y, 1)[0]
        slopes[i] = slope

    return slopes

test_df["hr_slope_60s"] = (
    test_df.groupby("encounter_id")["heart_rate"]
      .transform(lambda x: rolling_slope(x.values, window=12))
)

test_df.head()

,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index,hr_roll_mean_60s,spo2_roll_mean_60s,hr_roll_std_60s,hr_delta,spo2_delta,hr_slope_60s
7920,2025-01-02 06:00:00,00945edc-641d-5f9d-f4b0-eaf147a61556_11,102.0,112.0,82.0,16.0,97.0,0,30.0,129.333333,0.910714,102.000000,97.0,NaN,NaN,NaN,NaN
7921,2025-01-02 06:00:05,00945edc-641d-5f9d-f4b0-eaf147a61556_11,107.0,114.0,84.0,16.0,97.0,0,30.0,132.000000,0.938596,104.500000,97.0,3.535534,5.0,0.0,NaN
7922,2025-01-02 06:00:10,00945edc-641d-5f9d-f4b0-eaf147a61556_11,102.0,116.0,84.0,16.0,97.0,0,32.0,133.333333,0.879310,103.666667,97.0,2.886751,-5.0,0.0,NaN
7923,2025-01-02 06:00:15,00945edc-641d-5f9d-f4b0-eaf147a61556_11,98.0,118.0,86.0,16.0,97.0,0,32.0,136.000000,0.830508,102.250000,97.0,3.685557,-4.0,0.0,NaN
7924,2025-01-02 06:00:20,00945edc-641d-5f9d-f4b0-eaf147a61556_11,97.0,120.0,88.0,15.0,97.0,0,32.0,138.666667,0.808333,101.200000,97.0,3.962323,-1.0,0.0,NaN


In [37]:
train_df[["hr_delta", "spo2_delta", "hr_roll_std_60s", "hr_slope_60s"]] = \
train_df[["hr_delta", "spo2_delta", "hr_roll_std_60s", "hr_slope_60s"]].fillna(0)

train_df.head()

,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index,hr_roll_mean_60s,spo2_roll_mean_60s,hr_roll_std_60s,hr_delta,spo2_delta,hr_slope_60s
1577520,2025-04-03 02:00:00,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,75.0,110.0,58.0,15.0,97.0,0,52.0,112.000000,0.681818,75.000000,97.000000,0.000000,0.0,0.0,0.0
1577521,2025-04-03 02:00:05,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,78.0,110.0,58.0,13.0,98.0,0,52.0,112.000000,0.709091,76.500000,97.500000,2.121320,3.0,1.0,0.0
1577522,2025-04-03 02:00:10,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,83.0,110.0,58.0,15.0,97.0,0,52.0,112.000000,0.754545,78.666667,97.333333,4.041452,5.0,-1.0,0.0
1577523,2025-04-03 02:00:15,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,77.0,110.0,58.5,16.0,97.0,0,51.5,112.333333,0.700000,78.250000,97.250000,3.403430,-6.0,0.0,0.0
1577524,2025-04-03 02:00:20,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,79.0,108.0,58.0,17.0,97.0,0,50.0,110.666667,0.731481,78.400000,97.200000,2.966479,2.0,0.0,0.0


In [84]:
test_df[["hr_delta", "spo2_delta", "hr_roll_std_60s", "hr_slope_60s"]] = \
test_df[["hr_delta", "spo2_delta", "hr_roll_std_60s", "hr_slope_60s"]].fillna(0)

test_df.head()

,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index,hr_roll_mean_60s,spo2_roll_mean_60s,hr_roll_std_60s,hr_delta,spo2_delta,hr_slope_60s
7920,2025-01-02 06:00:00,00945edc-641d-5f9d-f4b0-eaf147a61556_11,102.0,112.0,82.0,16.0,97.0,0,30.0,129.333333,0.910714,102.000000,97.0,0.000000,0.0,0.0,0.0
7921,2025-01-02 06:00:05,00945edc-641d-5f9d-f4b0-eaf147a61556_11,107.0,114.0,84.0,16.0,97.0,0,30.0,132.000000,0.938596,104.500000,97.0,3.535534,5.0,0.0,0.0
7922,2025-01-02 06:00:10,00945edc-641d-5f9d-f4b0-eaf147a61556_11,102.0,116.0,84.0,16.0,97.0,0,32.0,133.333333,0.879310,103.666667,97.0,2.886751,-5.0,0.0,0.0
7923,2025-01-02 06:00:15,00945edc-641d-5f9d-f4b0-eaf147a61556_11,98.0,118.0,86.0,16.0,97.0,0,32.0,136.000000,0.830508,102.250000,97.0,3.685557,-4.0,0.0,0.0
7924,2025-01-02 06:00:20,00945edc-641d-5f9d-f4b0-eaf147a61556_11,97.0,120.0,88.0,15.0,97.0,0,32.0,138.666667,0.808333,101.200000,97.0,3.962323,-1.0,0.0,0.0


In [ ]:
train_df = train_df.drop(['timestamp', 'encounter_id'], axis=1)

In [85]:
test_df = test_df.drop(['timestamp', 'encounter_id'], axis=1)

In [65]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
cols = ['heart_rate', 'systolic_bp', 'diastolic_bp', 'respiratory_rate', 'oxygen_saturation', 'pulse_pressure', 'MAP', 'shock_index',	'hr_roll_mean_60s',	'spo2_roll_mean_60s',	'hr_roll_std_60s',	'hr_delta',	'spo2_delta',	'hr_slope_60s']

In [70]:
import numpy as np

train_df[train_df["shock_index"] == np.inf]
train_df["shock_index"] = train_df["shock_index"].replace([np.inf, -np.inf], np.nan)
median_value = train_df["shock_index"].median()
print(median_value)
train_df["shock_index"].fillna(median_value, inplace=True)

0.7407407407407407


C:\Users\sahil\AppData\Local\Temp\ipykernel_21472\3286412451.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df["shock_index"].fillna(median_value, inplace=True)


In [86]:
test_df[test_df["shock_index"] == np.inf]
test_df["shock_index"] = test_df["shock_index"].replace([np.inf, -np.inf], np.nan)
median_value = test_df["shock_index"].median()
print(median_value)
test_df["shock_index"].fillna(median_value, inplace=True)

0.7384615384615385


C:\Users\sahil\AppData\Local\Temp\ipykernel_21472\174319526.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_df["shock_index"].fillna(median_value, inplace=True)


In [73]:
train_df[cols] = sc.fit_transform(train_df[cols])

train_df.head()

,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index,hr_roll_mean_60s,spo2_roll_mean_60s,hr_roll_std_60s,hr_delta,spo2_delta,hr_slope_60s
1577520,-0.228943,-0.009705,-0.511248,-0.251163,0.325838,0,0.545307,-0.219912,-0.179413,-0.235804,0.330853,-0.935945,0.001353,0.004956,0.004585
1577521,-0.165939,-0.009705,-0.511248,-0.593834,0.370614,0,0.545307,-0.219912,-0.134543,-0.203517,0.353925,-0.642370,0.432478,0.257979,0.004585
1577522,-0.060933,-0.009705,-0.511248,-0.251163,0.325838,0,0.545307,-0.219912,-0.059759,-0.156881,0.346235,-0.376639,0.719895,-0.248067,0.004585
1577523,-0.186940,-0.009705,-0.488702,-0.079828,0.325838,0,0.520551,-0.210484,-0.149500,-0.165849,0.342389,-0.464936,-0.860898,0.004956,0.004585
1577524,-0.144938,-0.069509,-0.511248,0.091508,0.325838,0,0.446284,-0.257624,-0.097705,-0.162620,0.340082,-0.525407,0.288769,0.004956,0.004585


In [87]:
test_df[cols] = sc.fit_transform(test_df[cols])

test_df.head()

,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index,hr_roll_mean_60s,spo2_roll_mean_60s,hr_roll_std_60s,hr_delta,spo2_delta,hr_slope_60s
7920,0.366576,0.045101,0.562515,-0.074721,0.311754,0,-0.546053,0.264092,0.205437,0.374977,0.317409,-0.924275,0.001317,0.004559,0.004483
7921,0.474245,0.105764,0.653682,-0.074721,0.311754,0,-0.546053,0.340489,0.250777,0.430214,0.317409,-0.436979,0.730888,0.004559,0.004483
7922,0.366576,0.166428,0.653682,-0.074721,0.311754,0,-0.445431,0.378688,0.154371,0.411802,0.317409,-0.526399,-0.728254,0.004559,0.004483
7923,0.280441,0.227091,0.744848,-0.074721,0.311754,0,-0.445431,0.455085,0.075014,0.380501,0.317409,-0.416301,-0.582340,0.004559,0.004483
7924,0.258907,0.287754,0.836014,-0.248308,0.311754,0,-0.445431,0.531482,0.038955,0.357302,0.317409,-0.378155,-0.144597,0.004559,0.004483


In [105]:
# Create training and testing sets - also separate features/labels
X_train = train_df.drop(columns = ["label"])
y_train = train_df["label"]

X_test = test_df.drop(columns = ["label"])
y_test = test_df["label"]

In [106]:
from tensorflow.keras.utils import to_categorical

unique_states = sorted(y_train.unique())
num_classes = len(unique_states)

# Create a mapping from np.int to actual int
state_to_index = {state: i for i, state in enumerate(unique_states)}

# Convert our training labels to numerical indices
y_train_indices = y_train.map(state_to_index)

# One-hot encode the training labels
y_train_encoded = to_categorical(y_train_indices, num_classes=num_classes)
print("y_train_encoded shape:", y_train_encoded.shape, 
      "Example one-hot vector:", y_train_encoded[0])


# Repeat for testing data
unique_states = sorted(y_test.unique())
num_classes = len(unique_states)

state_to_index = {state: i for i, state in enumerate(unique_states)}

y_test_indices = y_test.map(state_to_index)

y_test_encoded = to_categorical(y_test_indices, num_classes=num_classes)
print("y_train_encoded shape:", y_test_encoded.shape, 
      "Example one-hot vector:", y_test_encoded[0])

y_train_encoded shape: (2109600, 4) Example one-hot vector: [1. 0. 0. 0.]
y_train_encoded shape: (451440, 4) Example one-hot vector: [1. 0. 0. 0.]


In [107]:
model = Sequential()

# Input layer: our features are the vitals
model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.2))

# Hidden layer 2
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))

# Hidden layer 3
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))

# Output layer for multi-class
model.add(Dense(num_classes, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 64)             │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,540 (37.27 KB)

 Trainable params: 9,540 (37.27 KB)

 Non-trainable params: 0 (0.00 B)

In [108]:
EPOCHS = 10
BATCH_SIZE = 512

history = model.fit(
    X_train,
    y_train_encoded,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,  # 90% train, 10% validation
    verbose=1
)

Epoch 1/10
3709/3709 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.7922 - loss: 0.5153 - val_accuracy: 0.8228 - val_loss: 0.4398
Epoch 2/10
3709/3709 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.8058 - loss: 0.4794 - val_accuracy: 0.8259 - val_loss: 0.4335
Epoch 3/10
3709/3709 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.8078 - loss: 0.4735 - val_accuracy: 0.8268 - val_loss: 0.4332
Epoch 4/10
3709/3709 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.8090 - loss: 0.4704 - val_accuracy: 0.8270 - val_loss: 0.4301
Epoch 5/10
3709/3709 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.8108 - loss: 0.4684 - val_accuracy: 0.8407 - val_loss: 0.4261
Epoch 6/10
3709/3709 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.8189 - loss: 0.4609 - val_accuracy: 0.8440 - val_loss: 0.4155
Epoch 7/10
3709/3709 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.8210 - loss: 0.4581 - val_accuracy: 0.8426 - val_loss: 0.4116
Epoch 8/10
3709/3709 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.8220 - loss: 0

In [91]:
model.evaluate(X_test, y_test_encoded)

14108/14108 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 0.6365 - loss: 1.3073


[1.3072736263275146, 0.6365453004837036]

In [109]:
y_pred = model.predict(X_test)

14108/14108 ━━━━━━━━━━━━━━━━━━━━ 17s 1ms/step


In [115]:
print(y_pred)

[[8.56993914e-01 1.19460933e-01 2.34372262e-02 1.07926426e-04]
 [9.15820301e-01 7.98648670e-02 4.31463588e-03 1.29595435e-07]
 [9.37252522e-01 5.98396547e-02 2.90773297e-03 4.97027379e-08]
 ...
 [9.60939944e-01 3.74501161e-02 1.60994136e-03 1.57278035e-09]
 [9.63270903e-01 3.52933891e-02 1.43568357e-03 8.54296078e-10]
 [9.68688071e-01 3.00273206e-02 1.28471327e-03 4.22765434e-10]]


In [111]:
import numpy as np

y_raghav = np.argmax(y_pred, axis=1)
y_raghav = y_raghav.tolist()

y_test_result = np.argmax(y_pred, axis=1)
y_test_result = y_test_result.tolist()

In [112]:
y_test_result

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,


In [113]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_result, y_raghav)

In [114]:
print(cm)

[[357619      0      0      0]
 [     0  52917      0      0]
 [     0      0  19001      0]
 [     0      0      0  21903]]
